In [ ]:
# Gerekli kütüphaneyi yükle
!pip install openai --quiet

In [ ]:
from openai import OpenAI
import os

# API anahtarını environment'tan al
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Model seçimi
MODEL = "gpt-4o-mini"


In [ ]:
def prompt_gonder(prompt, system_prompt=None, temperature=0.7):
    messages = []

    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature
    )

    return response.choices[0].message.content


---

## 1 Prompt Nedir?

**Prompt**, yapay zeka modeline verdiğimiz **talimat** veya **sorudur.**

Bunu şöyle düşünebilirsiniz:

| Gerçek Hayat | AI Dünyası |
|---|---|
| Garsondan sipariş vermek | AI'ya prompt yazmak |
| "Bir latte alabilir miyim?" | "Bana kahve çeşitlerini listele" |

Ne kadar **net** ve **açık** sipariş verirseniz, o kadar **doğru** sonuç alırsınız!

### Bir prompt'un temel bileşenleri:

```
┌─────────────────────────────────────┐
│   Talimat  → Ne yapmasını istiyorsun?    │
│   Bağlam   → Ek bilgi var mı?           │
│   Girdi    → İşlenecek veri nedir?       │
│   Çıktı    → Nasıl bir format istiyorsun?│
└─────────────────────────────────────┘
```

In [ ]:
# En basit prompt örneği
cevap = prompt_gonder("Python nedir?")
print(cevap)

Python, yüksek seviyeli, genel amaçlı bir programlama dilidir. 1991 yılında Guido van Rossum tarafından geliştirilmiştir. Python, okunabilirliği ön planda tutarak, basit ve anlaşılır bir sözdizimine sahiptir. Bu özellikleri sayesinde hem yeni başlayanlar hem de deneyimli programcılar için popüler bir seçim olmuştur.

Python'un bazı önemli özellikleri şunlardır:

1. **Okunabilirlik**: Python, kodun kolayca okunabilir olmasını sağlamak için tasarlanmıştır. Bu, geliştiricilerin kod üzerinde daha hızlı çalışmasına ve bakım yapmasına yardımcı olur.

2. **Geniş Kütüphane Desteği**: Python, çok sayıda standart kütüphane ve üçüncü taraf kütüphaneler sunar. Bu, farklı alanlardaki uygulamaları geliştirmeyi kolaylaştırır.

3. **Platform Bağımsızlığı**: Python, farklı işletim sistemlerinde çalışabilir; bu da geliştiricilerin kodlarını birden fazla platformda kullanabilmelerini sağlar.

4. **Yüksek Seviyeli Abstraksiyon**: Python, düşük seviyeli detaylarla uğraşmadan daha karmaşık problemlere odakl

##  Zero-Shot vs Few-Shot Prompting

###  Zero-Shot: Hiç örnek vermeden sormak
AI'ya herhangi bir örnek **vermeden** doğrudan görev veriyoruz.

###  Few-Shot: Birkaç örnek vererek sormak  
AI'ya birkaç **örnek gösterip** "bunlar gibi yap" diyoruz.

```
Zero-Shot:  "Bu cümlenin duygusunu analiz et"
Few-Shot:   "Örnekler: 'Harika!' → Pozitif, 'Berbat' → Negatif. Şimdi bu cümleyi analiz et"
```

###  Zero-Shot Örnekleri

In [ ]:
# ========================================
# ZERO-SHOT: Duygu Analizi (örnek vermeden)
# ========================================

prompt = """Aşağıdaki cümlenin duygusunu analiz et.
Sadece 'Pozitif', 'Negatif' veya 'Nötr' olarak cevap ver.

Cümle: "Bu ürün gerçekten harika, çok memnun kaldım!"
"""

cevap = prompt_gonder(prompt)
print(f"Zero-Shot Sonuç: {cevap}")

Zero-Shot Sonuç: Pozitif


###  Few-Shot Örnekleri

In [ ]:
# ========================================
# FEW-SHOT: Duygu Analizi (örneklerle)
# ========================================

prompt = """Aşağıdaki örneklere bakarak cümlenin duygusunu analiz et.

Örnekler:
- "Çok güzel bir deneyimdi, tekrar geleceğim!" → Pozitif
- "Paramı çöpe attım, kesinlikle tavsiye etmiyorum." → Negatif
- "Ürün dün geldi, kutusu maviydi." → Nötr
- "Harika lezzet, ailece bayıldık!" → Pozitif
- "Beklediğim gibi çıkmadı, hayal kırıklığı." → Negatif

Şimdi bu cümleyi analiz et:
"Fiyatına göre idare eder ama beklentilerimin altında kaldı."
"""

cevap = prompt_gonder(prompt, temperature=0)
print(f"Few-Shot Sonuç: {cevap}")

Few-Shot Sonuç: Cümlenin duygusu: Negatif

Açıklama: Cümlede "fiyatına göre idare eder" ifadesi bir miktar olumlu bir değerlendirme sunsa da, "ama beklentilerimin altında kaldı" kısmı, hayal kırıklığını ve olumsuz bir deneyimi ifade ediyor. Genel olarak, cümledeki olumsuzluk baskın olduğu için duygusu negatif olarak değerlendirilebilir.


###  Zero-Shot vs Few-Shot Karşılaştırma

In [ ]:
# ========================================
# KARŞILAŞTIRMA: Aynı görev, iki yaklaşım
# ========================================

test_cumle = "Siparişim 3 gün geç geldi ama ürünün kendisi fena değildi."

# --- Zero-Shot ---
zero_shot_prompt = f"""Bu cümlenin duygusunu analiz et ve 1-5 arası puan ver.
Format: Duygu: ... | Puan: .../5

Cümle: "{test_cumle}"""

# --- Few-Shot ---
few_shot_prompt = f"""Cümlelerin duygusunu analiz et ve 1-5 arası puan ver.

Örnekler:
"Harika ürün, bayıldım!" → Duygu: Pozitif | Puan: 5/5
"İdare eder, fena değil." → Duygu: Nötr | Puan: 3/5
"Çöpe attım, berbat kalite." → Duygu: Negatif | Puan: 1/5
"Geç geldi ama ürün güzeldi." → Duygu: Karışık | Puan: 3/5

Şimdi analiz et:
"{test_cumle}"""

print(" Test Cümlesi:")
print(f"   \"{test_cumle}\"\n")

print(" ZERO-SHOT Sonucu:")
print(f"   {prompt_gonder(zero_shot_prompt, temperature=0)}\n")

print(" FEW-SHOT Sonucu:")
print(f"   {prompt_gonder(few_shot_prompt, temperature=0)}")

 Test Cümlesi:
   "Siparişim 3 gün geç geldi ama ürünün kendisi fena değildi."

 ZERO-SHOT Sonucu:
   Duygu: Karışık (hayal kırıklığı ve memnuniyet) | Puan: 3/5

 FEW-SHOT Sonucu:
   Duygu: Karışık | Puan: 3/5


---

##  Prompt Tasarım Teknikleri

İyi bir prompt yazmak için kullanabileceğimiz teknikler:

| Teknik | Açıklama | Örnek |
|---|---|---|
|  Rol Verme | AI'ya bir karakter biç | "Sen bir aşçısın..." |
|  Format Belirleme | Çıktı formatını belirt | "JSON olarak döndür" |
|  Zincirleme Düşünme | Adım adım düşünmesini iste | "Adım adım açıkla" |
|  Kısıtlama Ekleme | Sınırlar koy | "Maksimum 50 kelime" |
|  Şablon Kullanma | Çıktı şablonu ver | "Başlık: ... Özet: ..." |

###  Teknik 1: Rol Verme (Role Prompting)

In [ ]:
# ========================================
# ROL VERME: Aynı soru, farklı roller
# ========================================

soru = "Yapay zeka nedir?"

roller = {
    " Anaokulu Öğretmeni": "Sen bir anaokulu öğretmenisin. Her şeyi 5 yaşındaki çocuklara anlatır gibi, çok basit ve eğlenceli bir dille açıklarsın.",
    " Yazılımcı": "Sen kıdemli bir yazılım mühendisisin. Teknik terimleri kullanarak, kod örnekleriyle açıklama yaparsın.",
    " Babaanne": "Sen teknolojiden hiç anlamayan sevecen bir babaannesin. Her şeyi mutfak ve günlük hayat örnekleriyle açıklarsın."
}

for rol_adi, system_prompt in roller.items():
    print(f"\n{'='*50}")
    print(f"{rol_adi}")
    print(f"{'='*50}")
    cevap = prompt_gonder(
        f"{soru} (2-3 cümleyle açıkla)",
        system_prompt=system_prompt,
        temperature=0.7
    )
    print(cevap)

###  Teknik 2: Format Belirleme

In [ ]:
# ========================================
# FORMAT BELİRLEME: JSON çıktı
# ========================================

prompt = """Aşağıdaki metinden bilgileri çıkar ve JSON formatında döndür.

Metin: "Ahmet Yılmaz, 28 yaşında, İstanbul'da yazılım mühendisi olarak
çalışıyor. Python ve JavaScript biliyor. Aylık maaşı 45.000 TL."

JSON formatı:
{
    "isim": "",
    "yas": 0,
    "sehir": "",
    "meslek": "",
    "yetenekler": [],
    "maas": ""
}
"""

cevap = prompt_gonder(prompt, temperature=0)
print(cevap)

###  Teknik 3: Zincirleme Düşünme (Chain of Thought)

In [ ]:
# ========================================
# ZİNCİRLEME DÜŞÜNME: Adım adım çözüm
# ========================================

# Düz soru (Chain of Thought olmadan)
duz_prompt = """Bir mağazada 3 tişört ve 2 pantolon aldım.
Tişörtler tanesi 150 TL, pantolonlar tanesi 300 TL.
%10 indirim uygulandı. Ne kadar ödedim?"""

# Chain of Thought ile
cot_prompt = """Bir mağazada 3 tişört ve 2 pantolon aldım.
Tişörtler tanesi 150 TL, pantolonlar tanesi 300 TL.
%10 indirim uygulandı. Ne kadar ödedim?

Lütfen adım adım düşün:
1. Önce tişörtlerin toplam fiyatını hesapla
2. Sonra pantolonların toplam fiyatını hesapla
3. Genel toplamı bul
4. İndirimi uygula
5. Son tutarı söyle"""

print(" DÜZ PROMPT:")
print(prompt_gonder(duz_prompt, temperature=0))
print("\n" + "="*50 + "\n")
print(" CHAIN OF THOUGHT:")
print(prompt_gonder(cot_prompt, temperature=0))

---

##  İyi vs Kötü Prompt

Şimdi aynı görev için **kötü** ve **iyi** promptları yan yana karşılaştıralım.

Her örnekte göreceğiz:
-  **Kötü prompt** → Belirsiz, genel, formatsız
-  **İyi prompt** → Spesifik, yapılandırılmış, bağlamlı

In [ ]:
# ========================================
# DEMO 1: E-posta Yazma
# ========================================

kotu_prompt = "Bir e-posta yaz."

iyi_prompt = """İş yerindeki müdürüme yıllık izin talebi için resmi bir e-posta yaz.

Detaylar:
- İzin tarihi: 15-22 Mart 2025
- Sebep: Aile ziyareti
- Müdürün adı: Mehmet Bey
- Benim adım: Ayşe

Ton: Resmi ama samimi
Uzunluk: Kısa ve öz (maksimum 100 kelime)
"""

print(" KÖTÜ PROMPT: 'Bir e-posta yaz.'")
print("-" * 40)
print(prompt_gonder(kotu_prompt))
print("\n" + "=" * 50 + "\n")
print(" İYİ PROMPT: (Detaylı talimat)")
print("-" * 40)
print(prompt_gonder(iyi_prompt))

 KÖTÜ PROMPT: 'Bir e-posta yaz.'
----------------------------------------
Tabii ki! Hangi konuyla ilgili bir e-posta yazmamı istersiniz? Alıcı, konu ve içerik hakkında biraz bilgi verirseniz, daha iyi yardımcı olabilirim.


 İYİ PROMPT: (Detaylı talimat)
----------------------------------------
Konu: Yıllık İzin Talebi

Sayın Mehmet Bey,

Umarım iyisinizdir. 15-22 Mart 2025 tarihleri arasında aile ziyareti sebebiyle yıllık izin talep ediyorum. Bu süre zarfında iş akışını aksatmamak adına gerekli düzenlemeleri yapmaya hazırım. 

İznimle ilgili onayınızı rica ederim. İlginiz için şimdiden teşekkür ederim.

Saygılarımla,  
Ayşe


---

## : Prompt Yazma Kontrol Listesi

Bir prompt yazarken kendinize şu soruları sorun:

```
 Görev net mi?           → Ne yapmasını istiyorum?
 Bağlam var mı?          → Arka plan bilgisi verdim mi?
 Format belirli mi?       → Çıktı nasıl görünmeli?
 Kısıtlamalar var mı?     → Uzunluk, dil, ton?
 Örnekler ekledim mi?     → Few-shot faydalı olur mu?
 Rol tanımladım mı?       → Kim gibi davranmalı?
```


> **"AI bir düşünce okuyucusu değil. Ne kadar net olursan, o kadar iyi sonuç alırsın."**